In [1]:
import pandas as pd
import numpy as np

In [2]:
df_raw = pd.read_csv('../../dataset/original/cc_calls.csv')
print(len(df_raw.columns))
print(df_raw.info())
print(df_raw.head())

33
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32882 entries, 0 to 32881
Data columns (total 33 columns):
 #   Column                                    Non-Null Count  Dtype  
---  ------                                    --------------  -----  
 0   Contact_ID                                32882 non-null  float64
 1   Call_Date                                 32882 non-null  object 
 2   Direction                                 32882 non-null  object 
 3   cc_care_package                           32744 non-null  object 
 4   cc_care_package_discussed                 32744 non-null  object 
 5   cc_urgency_getting_on_site                32744 non-null  object 
 6   cc_external_consultant                    32744 non-null  object 
 7   cc_agent_cross_sell_attempt               32744 non-null  object 
 8   cc_customer_issues_concerns               32744 non-null  object 
 9   cc_business_struggles_financial_hardship  32744 non-null  object 
 10  cc_call_initiated_by           

In [3]:
df_cols = df_raw.columns
print(df_cols)
# //save as a csv file
df_cols_df = pd.DataFrame(df_cols, columns=['Columns'])
df_cols_df.to_csv('../../dataset/01_raw/cc_calls/columns.csv', index=True)

Index(['Contact_ID', 'Call_Date', 'Direction', 'cc_care_package',
       'cc_care_package_discussed', 'cc_urgency_getting_on_site',
       'cc_external_consultant', 'cc_agent_cross_sell_attempt',
       'cc_customer_issues_concerns',
       'cc_business_struggles_financial_hardship', 'cc_call_initiated_by',
       'cc_questionnaire_completion', 'cc_chasing_response',
       'cc_issues_within_questionnaire', 'cc_login_issues',
       'cc_platform_issues', 'cc_dissatisfaction_time_to_complete',
       'cc_process_complexity_concerns', 'cc_questions_harder_than_expected',
       'cc_dissatisfaction_support', 'cc_contractor_sentiment',
       'cc_contractor_sentiment_start_score',
       'cc_contractor_sentiment_end_score',
       'cc_contractor_sentiment_overall_score',
       'cc_contractor_sentiment_issues_score', 'cc_pricing_mentioned',
       'cc_pricing_sentiment_impact', 'cc_refund_discussed',
       'cc_contractor_suggest_leave', 'cc_contractor_complained', 'Co_Ref',
       'Analys

## Analysing the Null Cols

In [4]:
print(df_raw.isnull().sum())

Contact_ID                                     0
Call_Date                                      0
Direction                                      0
cc_care_package                              138
cc_care_package_discussed                    138
cc_urgency_getting_on_site                   138
cc_external_consultant                       138
cc_agent_cross_sell_attempt                  138
cc_customer_issues_concerns                  138
cc_business_struggles_financial_hardship     138
cc_call_initiated_by                         138
cc_questionnaire_completion                   32
cc_chasing_response                           32
cc_issues_within_questionnaire               466
cc_login_issues                               33
cc_platform_issues                            33
cc_dissatisfaction_time_to_complete           33
cc_process_complexity_concerns                38
cc_questions_harder_than_expected             37
cc_dissatisfaction_support                    36
cc_contractor_sentim

In [5]:
null_columns = df_raw.columns[df_raw.isnull().any()]
print("Columns with null values:", null_columns)

#save the null cols as a csv file
null_cols_df = pd.DataFrame(null_columns, columns=['Null Columns']) 
null_cols_df.to_csv('../../dataset/01_raw/cc_calls/null_columns.csv', index=True)


Columns with null values: Index(['cc_care_package', 'cc_care_package_discussed',
       'cc_urgency_getting_on_site', 'cc_external_consultant',
       'cc_agent_cross_sell_attempt', 'cc_customer_issues_concerns',
       'cc_business_struggles_financial_hardship', 'cc_call_initiated_by',
       'cc_questionnaire_completion', 'cc_chasing_response',
       'cc_issues_within_questionnaire', 'cc_login_issues',
       'cc_platform_issues', 'cc_dissatisfaction_time_to_complete',
       'cc_process_complexity_concerns', 'cc_questions_harder_than_expected',
       'cc_dissatisfaction_support', 'cc_contractor_sentiment',
       'cc_contractor_sentiment_start_score',
       'cc_contractor_sentiment_end_score',
       'cc_contractor_sentiment_overall_score',
       'cc_contractor_sentiment_issues_score', 'cc_pricing_mentioned',
       'cc_pricing_sentiment_impact', 'cc_refund_discussed',
       'cc_contractor_suggest_leave', 'cc_contractor_complained', 'Co_Ref'],
      dtype='object')


## Getting the datatype of each columns and uniques values if (more than 10 values then as ranges)

In [6]:



def is_date_column(series):
    return pd.api.types.is_datetime64_any_dtype(series)


def is_coref_column(series, threshold=50):
    num_unique = series.nunique(dropna=True)
    return series.dtype == 'object' and num_unique > threshold


def classify_column(series):
    if pd.api.types.is_datetime64_any_dtype(series):
        return "datetime"

    if pd.api.types.is_numeric_dtype(series):
        return "numerical"

    if series.dtype == 'object':
        # Try parsing to datetime
        try:
            parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
            if all(parsed.dt.time == pd.Timestamp("00:00:00").time()):
                return "date"
            else:
                return "datetime"
        except:
            pass

        # Try parsing to time
        try:
            pd.to_datetime(series.dropna().head(10), format='%H:%M:%S', errors='raise')
            return "time"
        except:
            pass

        return "categorical"

    return "categorical"


def get_unique_representation(series):
    unique_vals = series.dropna().unique()
    num_unique = len(unique_vals)

    if is_date_column(series) or is_coref_column(series):
        return []

    if pd.api.types.is_numeric_dtype(series):
        if num_unique <= 10:
            return sorted(unique_vals.tolist())
        else:
            return f"{series.min()} - {series.max()}"

    return unique_vals.tolist()


def create_data_validation_df(df):
    summary = []

    for col in df.columns:
        series = df[col]

        col_summary = {
            "column_name": col,
            "data_type": str(series.dtype),
            "column_type": classify_column(series),
            "unique_values": get_unique_representation(series),
            "null_count": series.isna().sum(),
            "num_unique": series.nunique(dropna=True)
        }

        summary.append(col_summary)

    return pd.DataFrame(summary)


def save_validation_report(df, output_path):
    dt_val_df = create_data_validation_df(df)
    print(dt_val_df)

    dt_val_df.to_csv(output_path, index=True)
    print(f"\nReport saved to: {output_path}")


# output_file = './dataset/01_raw/ad_hoc/data_validation_summary.csv'
# save_validation_report(df_raw, output_file)

In [7]:
def create_data_validation_df(df):
    summary = []

    for col in df.columns:
        series = df[col]

        col_summary = {
            "column_name": col,
            "data_type": str(series.dtype),
            "column_type": classify_column(series),
            "null_count": series.isna().sum(),
            "null_percentage": round((series.isna().sum() / len(df)) * 100, 2),
            "num_unique": series.nunique(dropna=True),
            "unique_values": get_unique_representation(series),
            "top_values": series.value_counts(dropna=True).head(5).to_dict(),
            "skewness": series.skew() if pd.api.types.is_numeric_dtype(series) else None,
            "constant_column": series.nunique(dropna=True) <= 1
        }

        summary.append(col_summary)

    return pd.DataFrame(summary)

df_validation_df = create_data_validation_df(df_raw)
print(df_validation_df.head())
df_validation_df.to_csv('../../dataset/01_raw/cc_calls/data_validation_summary.csv', index=True)




C:\Users\Asus\AppData\Local\Temp\ipykernel_22584\3920180566.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_22584\3920180566.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_22584\3920180566.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_

                 column_name data_type  column_type  null_count  \
0                 Contact_ID   float64    numerical           0   
1                  Call_Date    object  categorical           0   
2                  Direction    object  categorical           0   
3            cc_care_package    object  categorical         138   
4  cc_care_package_discussed    object  categorical         138   

   null_percentage  num_unique                    unique_values  \
0             0.00         496  193615000000.0 - 691835000000.0   
1             0.00         323                               []   
2             0.00           2            [OUT_BOUND, IN_BOUND]   
3             0.42          57                               []   
4             0.42           3              [Yes, No, [Yes/No]]   

                                          top_values  skewness  \
0  {685446000000.0: 289, 691479000000.0: 282, 691... -2.004115   
1  {'20-10-2025': 298, '05-01-2026': 282, '06-01-...       NaN

C:\Users\Asus\AppData\Local\Temp\ipykernel_22584\3920180566.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_22584\3920180566.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_22584\3920180566.py:20: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed = pd.to_datetime(series.dropna().head(10), errors='raise')
C:\Users\Asus\AppData\Local\Temp\ipykernel_

In [8]:
def check_mixed_data_types(df):
    mixed_summary = []

    for col in df.columns:
        series = df[col].dropna()

        # Try numeric conversion
        converted = pd.to_numeric(series, errors='coerce')

        # Count valid numeric vs invalid
        numeric_count = converted.notna().sum()
        non_numeric_count = converted.isna().sum()

        # If both exist → mixed
        if numeric_count > 0 and non_numeric_count > 0:
            sample_numeric = series[converted.notna()].iloc[0]
            sample_non_numeric = series[converted.isna()].iloc[0]

            mixed_summary.append({
                "column": col,
                "numeric_count": int(numeric_count),
                "non_numeric_count": int(non_numeric_count),
                "sample_numeric": sample_numeric,
                "sample_non_numeric": sample_non_numeric
            })

    return pd.DataFrame(mixed_summary)

mixed_df = check_mixed_data_types(df_raw)
print(mixed_df)

mixed_df.to_csv('../../dataset/01_raw/cc_calls/mixed_data_types.csv', index=False)

                                  column  numeric_count  non_numeric_count  \
0    cc_contractor_sentiment_start_score          32638                149   
1      cc_contractor_sentiment_end_score          32639                148   
2  cc_contractor_sentiment_overall_score          27896               4891   
3   cc_contractor_sentiment_issues_score          11376              21411   
4                   cc_pricing_mentioned             40              32747   
5            cc_pricing_sentiment_impact             17              32770   
6                    cc_refund_discussed              8              32779   

  sample_numeric                                 sample_non_numeric  
0             20                                            Neutral  
1             30   which was caused by an expired SSIP provider ...  
2             30                                      Not Discussed  
3             20                                      Not Discussed  
4             80         

## =================================================================================